# 4 · Agente

**Módulo 1 · Introducción.** El proyecto del módulo es un agente de cálculo simbólico. El router de la lección 3 deja decidir al modelo una sola vez y termina con el resultado de la herramienta, que el modelo nunca lee. Esta lección cambia una arista, $\texttt{tools} \to \texttt{END}$ por $\texttt{tools} \to \texttt{modelo}$: el modelo lee cada resultado y vuelve a decidir, hasta que responde sin pedidos. Ese grafo es el agente.

La arista nueva cierra un ciclo, y con el ciclo vuelve la pregunta de la lección 1: ¿la ejecución termina? Se define el agente, se cuentan sus supersteps, se prueba que ninguna variante garantiza que termine y se le agrega una guardia con `RemainingSteps`, que lo hace llegar a `END` por sí mismo, sin `GraphRecursionError`.

**Dónde corre.** Las secciones 1 a 5 y la profundización no usan el modelo y corren en el PC, con modelos guionados. La sección 6 usa `qwen3:8b`, que corre en Kaggle con una GPU; el [índice del módulo](README.md) explica cómo abrir el notebook allí.

Requisito: `pip install "langgraph>=1.2" "langchain-ollama>=1.1" "sympy>=1.14"`. Las salidas de este notebook son de langgraph 1.2.17, langgraph-prebuilt 1.1.1, langchain-core 1.7.0, langchain-ollama 1.1.0, sympy 1.14.0 y Ollama 0.40.13 con `qwen3:8b`, en una GPU T4 de Kaggle.

**Preparación.** En Kaggle, la celda siguiente instala las dependencias y Ollama, arranca el servidor y descarga `qwen3:8b`, en poco más de un minuto. En el PC no hace nada.

In [ ]:
import os
import subprocess
import time
import urllib.request

EN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ


def correr(comando: str) -> None:
    """Corre un comando de shell y, si falla, muestra el final de su error."""
    r = subprocess.run(comando, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"falló: {comando}\n{r.stderr[-2000:]}")


def preparar_kaggle() -> None:
    """Instala las dependencias y Ollama, arranca el servidor y descarga qwen3:8b."""
    correr("pip install -q 'langgraph==1.2.17' 'langgraph-prebuilt==1.1.1' 'langchain-core==1.7.0'"
           " 'langchain-ollama==1.1.0' 'sympy==1.14.0'")
    correr("(command -v zstd || (apt-get -qq update && apt-get -qq install -y zstd))"
           " && curl -fsSL https://ollama.com/install.sh | sh")
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/version")
            break
        except OSError:
            time.sleep(1)
    else:
        raise RuntimeError("el servidor de Ollama no respondió en 60 s")
    correr("ollama pull qwen3:8b")


if EN_KAGGLE:
    preparar_kaggle()

In [ ]:
from importlib.metadata import version

for paquete in ("langgraph", "langgraph-prebuilt", "langchain-core", "langchain-ollama", "sympy"):
    print(paquete, version(paquete))